In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model="claude-haiku-4-5"

### Multi-turn conversation

The Messages API is stateless, so a "conversation" is just a growing list of `{role, content}`
messages that we resend on every request. This lets Claude ask a clarifying question (e.g. "which
order?") and then use the user's follow-up answer in the next turn.

We keep the conversation in a simple Python list and append both the user's and Claude's turns to it.

In [2]:
SYSTEM_PROMPT = """You are a helpful shop assistant for an online store.
When a customer asks for an order return or refund, you must first ask for
their order number before you can proceed, unless they've already given it.
Once you have the order number, confirm the return and explain the next steps."""

# The whole conversation lives in this list — we resend it on every call.
messages = []


def send_message(user_text: str) -> str:
    """Append a user turn, call Claude, append its reply, and return the reply text."""
    messages.append({"role": "user", "content": user_text})

    response = client.messages.create(
        model=model,
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=messages,
    )

    reply = next(block.text for block in response.content if block.type == "text")
    messages.append({"role": "assistant", "content": reply})
    return reply

In [3]:
# Turn 1: vague request — Claude should ask a clarifying question (the order number)
print("User: I want to return my order")
print("Claude:", send_message("I want to return my order"))

User: I want to return my order
Claude: I'd be happy to help you with your return! 

To get started, could you please provide me with your **order number**? You can find this in your confirmation email or in your account under "Order History."

Once I have that, I'll be able to confirm the details and explain the next steps for your return.


In [4]:
# Turn 2: the clarifying answer — Claude now has enough context from the earlier turns
print("User: It's order #A1234")
print("Claude:", send_message("It's order #A1234"))

User: It's order #A1234
Claude: Thank you! I have your order number: **#A1234**.

I can confirm that we're ready to process your return. Here are the next steps:

1. **Return Authorization**: Your return has been confirmed for order #A1234
2. **Return Shipping**: You'll receive a prepaid return shipping label via email shortly (check your inbox and spam folder)
3. **Pack Your Item**: Please pack the item securely in its original packaging if possible
4. **Ship It Back**: Use the provided label to send it back to us
5. **Refund Processing**: Once we receive and inspect your return, we'll process your refund within 5-7 business days

**A few things to note:**
- Please keep your tracking number for reference
- Make sure the item is in resellable condition
- If you have any questions while packing, feel free to reach out

Is there anything else you'd like to know about the return process, or do you have any questions about the item you're returning?


In [5]:
# The full conversation history that was (re)sent on every request
messages

[{'role': 'user', 'content': 'I want to return my order'},
 {'role': 'assistant',
  'content': 'I\'d be happy to help you with your return! \n\nTo get started, could you please provide me with your **order number**? You can find this in your confirmation email or in your account under "Order History."\n\nOnce I have that, I\'ll be able to confirm the details and explain the next steps for your return.'},
 {'role': 'user', 'content': "It's order #A1234"},
 {'role': 'assistant',
  'content': "Thank you! I have your order number: **#A1234**.\n\nI can confirm that we're ready to process your return. Here are the next steps:\n\n1. **Return Authorization**: Your return has been confirmed for order #A1234\n2. **Return Shipping**: You'll receive a prepaid return shipping label via email shortly (check your inbox and spam folder)\n3. **Pack Your Item**: Please pack the item securely in its original packaging if possible\n4. **Ship It Back**: Use the provided label to send it back to us\n5. **Re